# 01 — Coleta e consolidação dos dados

## Objetivo do notebook

Este notebook realiza a leitura, a validação estrutural e a consolidação dos dados históricos horários da estação meteorológica automática **A701 (São Paulo – Mirante)** do INMET, referentes ao período de **2010 a 2025**.

Os arquivos originais são preservados na camada `bronze`. A base consolidada gerada aqui é gravada na camada `silver`; o tratamento de qualidade (incluindo os códigos `-9999`) e os dados finais serão produzidos posteriormente nas camadas `silver`/`gold`.

## Bibliotecas e caminhos

Confirmação do interpretador Python do ambiente virtual e importação das bibliotecas utilizadas no notebook. O caminho da camada `bronze`, onde estão os arquivos originais, é definido na seção *Identificação dos arquivos*.

In [1]:
import sys

print(sys.executable)

C:\Users\arthur.pereira\Desktop\tcc_inmet_A701\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import statsmodels

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


In [3]:
def ler_csv_bronze(arquivo, remover_colunas_vazias=False):
    """Lê um arquivo horário da estação na camada bronze.

    Usa o mesmo padrão de leitura em todo o notebook: separador ";",
    8 linhas de metadados ignoradas, encoding "latin1" e vírgula como
    separador decimal. Quando ``remover_colunas_vazias`` é True, descarta
    as colunas totalmente vazias geradas pelo ";" ao final de cada linha.
    """
    df = pd.read_csv(
        arquivo,
        sep=";",
        skiprows=8,
        encoding="latin1",
        decimal=","
    )

    if remover_colunas_vazias:
        df = df.dropna(axis=1, how="all")

    return df

## Identificação dos arquivos

Localização dos arquivos `.csv` da estação A701 na camada `bronze` e listagem dos arquivos encontrados.

In [4]:
from pathlib import Path

pasta_bronze = Path("../data/bronze")

arquivos = sorted(pasta_bronze.glob("*.csv"))

print(f"Quantidade de arquivos encontrados: {len(arquivos)}")

for arquivo in arquivos:
    print(arquivo.name)

Quantidade de arquivos encontrados: 16
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2010_A_31-12-2010.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2011_A_31-12-2011.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2012_A_31-12-2012.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2013_A_31-12-2013.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2014_A_31-12-2014.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2015_A_31-12-2015.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2016_A_31-12-2016.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2017_A_31-12-2017.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2018_A_31-12-2018.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2019_A_31-12-2019.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2020_A_31-12-2020.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2021_A_31-12-2021.csv
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2022_A_31-12-2022.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2023_A_31-12-2023.CSV
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2024_A_3

## Validação dos anos

Extração do ano de cada arquivo pelo padrão do nome e verificação de que o período de 2010 a 2025 está completo, sem anos faltantes ou fora do intervalo esperado.

In [5]:
import re

anos_encontrados = []

for arquivo in arquivos:
    match = re.search(r"01-01-(\d{4})_A_31-12-(\d{4})", arquivo.name)

    if match:
        ano_inicio = int(match.group(1))
        ano_fim = int(match.group(2))

        anos_encontrados.append(ano_inicio)

        print(
            f"{arquivo.name} -> "
            f"início: {ano_inicio} | fim: {ano_fim}"
        )
    else:
        print(f"Não foi possível identificar o ano: {arquivo.name}")

INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2010_A_31-12-2010.CSV -> início: 2010 | fim: 2010
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2011_A_31-12-2011.CSV -> início: 2011 | fim: 2011
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2012_A_31-12-2012.CSV -> início: 2012 | fim: 2012
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2013_A_31-12-2013.CSV -> início: 2013 | fim: 2013
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2014_A_31-12-2014.CSV -> início: 2014 | fim: 2014
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2015_A_31-12-2015.CSV -> início: 2015 | fim: 2015
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2016_A_31-12-2016.CSV -> início: 2016 | fim: 2016
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2017_A_31-12-2017.CSV -> início: 2017 | fim: 2017
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2018_A_31-12-2018.CSV -> início: 2018 | fim: 2018
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2019_A_31-12-2019.CSV -> início: 2019 | fim: 2019
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2020_A_31-12-2020.CSV -> in

In [6]:
anos_esperados = set(range(2010, 2026))
anos_encontrados = set(anos_encontrados)

faltantes = sorted(anos_esperados - anos_encontrados)
extras = sorted(anos_encontrados - anos_esperados)

print(f"Quantidade de anos encontrados: {len(anos_encontrados)}")
print(f"Anos faltantes: {faltantes}")
print(f"Anos fora do período esperado: {extras}")

Quantidade de anos encontrados: 16
Anos faltantes: []
Anos fora do período esperado: []


## Inspeção de um arquivo

Inspeção de um arquivo representativo para entender o cabeçalho (metadados nas primeiras linhas), o separador, o *encoding* e o separador decimal, além de verificar o tipo e a quantidade das variáveis.

Nesta etapa os códigos `-9999` são apenas **contabilizados**, para dimensionar a ausência de dados; o tratamento desses valores será feito em etapa posterior.

In [7]:
arquivo_teste = arquivos[0]

print(arquivo_teste.name)


with open(arquivo_teste, "r", encoding="latin1") as f:
    for _ in range(12):
        print(f.readline().strip())

INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2010_A_31-12-2010.CSV
REGIÃO:;SE
UF:;SP
ESTAÇÃO:;SAO PAULO - MIRANTE
CODIGO (WMO):;A701
LATITUDE:;-23,48333333
LONGITUDE:;-46,61666666
ALTITUDE:;792,06
DATA DE FUNDAÇÃO (YYYY-MM-DD):;2006-07-25
DATA (YYYY-MM-DD);HORA (UTC);PRECIPITAÇÃO TOTAL, HORÁRIO (mm);PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB);PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB);PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB);RADIACAO GLOBAL (KJ/m²);TEMPERATURA DO AR - BULBO SECO, HORARIA (°C);TEMPERATURA DO PONTO DE ORVALHO (°C);TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C);TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C);TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C);TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C);UMIDADE REL. MAX. NA HORA ANT. (AUT) (%);UMIDADE REL. MIN. NA HORA ANT. (AUT) (%);UMIDADE RELATIVA DO AR, HORARIA (%);VENTO, DIREÇÃO HORARIA (gr) (° (gr));VENTO, RAJADA MAXIMA (m/s);VENTO, VELOCIDADE HORARIA (m/s);
2010-01-01;00:00;0;923,8;923,9;923,3;-9999;22

In [8]:
df_teste = ler_csv_bronze(arquivo_teste)

df_teste.head()

,DATA (YYYY-MM-DD),HORA (UTC),"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (KJ/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19
0,2010-01-01,00:00,0.0,923.8,923.9,923.3,-9999.0,22.2,18.5,22.2,21.9,19.0,18.3,83,79,79,155,7.5,2.8,NaN
1,2010-01-01,01:00,0.6,924.4,924.4,923.8,-9999.0,21.1,19.2,22.3,21.0,19.3,18.1,90,77,89,137,8.2,2.0,NaN
2,2010-01-01,02:00,0.0,924.4,924.5,924.4,-9999.0,21.6,18.9,21.6,21.2,19.3,18.8,89,84,85,148,6.8,2.1,NaN
3,2010-01-01,03:00,0.0,923.9,924.4,923.9,-9999.0,21.7,18.5,21.8,21.5,18.9,18.4,85,81,82,137,7.4,2.3,NaN
4,2010-01-01,04:00,0.0,923.0,923.9,923.0,-9999.0,21.4,18.2,21.7,21.4,18.5,18.2,83,82,82,135,7.8,2.9,NaN


In [9]:
print(f"Linhas: {df_teste.shape[0]}")
print(f"Colunas: {df_teste.shape[1]}")

df_teste.info()

Linhas: 8760
Colunas: 20
<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 20 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   DATA (YYYY-MM-DD)                                      8760 non-null   str    
 1   HORA (UTC)                                             8760 non-null   str    
 2   PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                       8760 non-null   float64
 3   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  8760 non-null   float64
 4   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)        8760 non-null   float64
 5   PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)       8760 non-null   float64
 6   RADIACAO GLOBAL (KJ/m²)                                8760 non-null   float64
 7   TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)           8760 non-null   float64
 8   TEMPERATURA DO PONTO DE ORVALHO (°

In [10]:
df_teste.columns.tolist()

['DATA (YYYY-MM-DD)',
 'HORA (UTC)',
 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
 'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
 'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
 'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
 'RADIACAO GLOBAL (KJ/m²)',
 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
 'TEMPERATURA DO PONTO DE ORVALHO (°C)',
 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
 'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
 'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
 'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
 'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
 'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
 'UMIDADE RELATIVA DO AR, HORARIA (%)',
 'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
 'VENTO, RAJADA MAXIMA (m/s)',
 'VENTO, VELOCIDADE HORARIA (m/s)',
 'Unnamed: 19']

In [11]:
df_teste = df_teste.drop(columns=["Unnamed: 19"])

print(df_teste.shape)
df_teste.head()

(8760, 19)


,DATA (YYYY-MM-DD),HORA (UTC),"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (KJ/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)"
0,2010-01-01,00:00,0.0,923.8,923.9,923.3,-9999.0,22.2,18.5,22.2,21.9,19.0,18.3,83,79,79,155,7.5,2.8
1,2010-01-01,01:00,0.6,924.4,924.4,923.8,-9999.0,21.1,19.2,22.3,21.0,19.3,18.1,90,77,89,137,8.2,2.0
2,2010-01-01,02:00,0.0,924.4,924.5,924.4,-9999.0,21.6,18.9,21.6,21.2,19.3,18.8,89,84,85,148,6.8,2.1
3,2010-01-01,03:00,0.0,923.9,924.4,923.9,-9999.0,21.7,18.5,21.8,21.5,18.9,18.4,85,81,82,137,7.4,2.3
4,2010-01-01,04:00,0.0,923.0,923.9,923.0,-9999.0,21.4,18.2,21.7,21.4,18.5,18.2,83,82,82,135,7.8,2.9


In [12]:
contagem_9999 = (df_teste == -9999).sum()

contagem_9999[contagem_9999 > 0].sort_values(ascending=False)

RADIACAO GLOBAL (KJ/m²)                                  4046
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)              162
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)                 63
VENTO, RAJADA MAXIMA (m/s)                                 13
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                   12
TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)                 12
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)           12
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)           12
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)            12
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                   12
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)           12
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                           10
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)      10
TEMPERATURA DO PONTO DE ORVALHO (°C)                       10
UMIDADE RELATIVA DO AR, HORARIA (%)                        10
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                       10
VENTO, V

## Validação estrutural

Leitura de todos os arquivos anuais para comparar o número de linhas e colunas e verificar se os nomes das colunas se mantêm iguais ao longo do período, identificando eventuais diferenças de nomenclatura.

In [13]:
estrutura_arquivos = []

for arquivo in arquivos:
    # Lê o arquivo e remove colunas totalmente vazias criadas pelo ";" final
    df = ler_csv_bronze(arquivo, remover_colunas_vazias=True)

    estrutura_arquivos.append({
        "arquivo": arquivo.name,
        "linhas": df.shape[0],
        "colunas": df.shape[1],
        "nomes_colunas": list(df.columns)
    })

for item in estrutura_arquivos:
    print(
        f"{item['arquivo']} | "
        f"{item['linhas']} linhas | "
        f"{item['colunas']} colunas"
    )

INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2010_A_31-12-2010.CSV | 8760 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2011_A_31-12-2011.CSV | 8760 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2012_A_31-12-2012.CSV | 8784 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2013_A_31-12-2013.CSV | 8760 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2014_A_31-12-2014.CSV | 8760 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2015_A_31-12-2015.CSV | 8760 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2016_A_31-12-2016.CSV | 8784 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2017_A_31-12-2017.CSV | 8760 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2018_A_31-12-2018.CSV | 8760 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2019_A_31-12-2019.CSV | 8760 linhas | 19 colunas
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2020_A_31-12-2020.CSV | 8784 linhas |

In [14]:
colunas_referencia = estrutura_arquivos[0]["nomes_colunas"]

for item in estrutura_arquivos:
    iguais = item["nomes_colunas"] == colunas_referencia

    print(
        f"{item['arquivo']} -> "
        f"estrutura igual a 2010: {iguais}"
    )

INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2010_A_31-12-2010.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2011_A_31-12-2011.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2012_A_31-12-2012.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2013_A_31-12-2013.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2014_A_31-12-2014.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2015_A_31-12-2015.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2016_A_31-12-2016.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2017_A_31-12-2017.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2018_A_31-12-2018.CSV -> estrutura igual a 2010: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2019_A_31-12-2019.CSV -> estrutura igual a 2010: False
INMET_SE_SP_A701_SAO PAULO - 

In [15]:
colunas_2010 = set(estrutura_arquivos[0]["nomes_colunas"])

for item in estrutura_arquivos:
    colunas_atual = set(item["nomes_colunas"])

    faltando = colunas_2010 - colunas_atual
    novas = colunas_atual - colunas_2010

    if faltando or novas:
        print(f"\nArquivo: {item['arquivo']}")
        print("Colunas que existiam em 2010 e não aparecem aqui:")
        print(faltando)

        print("Colunas novas/diferentes:")
        print(novas)


Arquivo: INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2019_A_31-12-2019.CSV
Colunas que existiam em 2010 e não aparecem aqui:
{'HORA (UTC)', 'DATA (YYYY-MM-DD)'}
Colunas novas/diferentes:
{'Hora UTC', 'Data'}

Arquivo: INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2020_A_31-12-2020.CSV
Colunas que existiam em 2010 e não aparecem aqui:
{'HORA (UTC)', 'DATA (YYYY-MM-DD)', 'RADIACAO GLOBAL (KJ/m²)'}
Colunas novas/diferentes:
{'RADIACAO GLOBAL (Kj/m²)', 'Hora UTC', 'Data'}

Arquivo: INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2021_A_31-12-2021.csv
Colunas que existiam em 2010 e não aparecem aqui:
{'HORA (UTC)', 'DATA (YYYY-MM-DD)', 'RADIACAO GLOBAL (KJ/m²)'}
Colunas novas/diferentes:
{'RADIACAO GLOBAL (Kj/m²)', 'Hora UTC', 'Data'}

Arquivo: INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2022_A_31-12-2022.CSV
Colunas que existiam em 2010 e não aparecem aqui:
{'HORA (UTC)', 'DATA (YYYY-MM-DD)', 'RADIACAO GLOBAL (KJ/m²)'}
Colunas novas/diferentes:
{'RADIACAO GLOBAL (Kj/m²)', 'Hora UTC', 'Data'}

Arqui

## Padronização do esquema

A validação estrutural mostrou que todos os arquivos anuais possuem 19 variáveis. Entretanto, foram identificadas alterações na nomenclatura de algumas colunas ao longo do período.

A partir de 2019, os campos `DATA (YYYY-MM-DD)` e `HORA (UTC)` passaram a ser apresentados como `Data` e `Hora UTC`. A partir de 2020, também foi identificada uma alteração na capitalização da unidade da variável de radiação global, de `KJ/m²` para `Kj/m²`.

Como essas diferenças representam apenas mudanças de nomenclatura e não alterações no significado das variáveis, os nomes das colunas são padronizados durante a leitura dos arquivos. Os arquivos originais permanecem preservados na camada `bronze`.

In [16]:
mapa_colunas = {
    "Data": "DATA (YYYY-MM-DD)",
    "Hora UTC": "HORA (UTC)",
    "RADIACAO GLOBAL (Kj/m²)": "RADIACAO GLOBAL (KJ/m²)"
}

In [17]:
estrutura_padronizada = []

for arquivo in arquivos:
    # Lê o arquivo e remove a coluna vazia criada pelo ";" final
    df = ler_csv_bronze(arquivo, remover_colunas_vazias=True)

    # Padroniza nomes que mudaram ao longo dos anos
    df = df.rename(columns=mapa_colunas)

    estrutura_padronizada.append({
        "arquivo": arquivo.name,
        "linhas": df.shape[0],
        "colunas": df.shape[1],
        "nomes_colunas": list(df.columns)
    })

In [18]:
colunas_referencia = estrutura_padronizada[0]["nomes_colunas"]

for item in estrutura_padronizada:
    iguais = item["nomes_colunas"] == colunas_referencia

    print(
        f"{item['arquivo']} -> "
        f"estrutura padronizada: {iguais}"
    )

INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2010_A_31-12-2010.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2011_A_31-12-2011.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2012_A_31-12-2012.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2013_A_31-12-2013.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2014_A_31-12-2014.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2015_A_31-12-2015.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2016_A_31-12-2016.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2017_A_31-12-2017.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2018_A_31-12-2018.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2019_A_31-12-2019.CSV -> estrutura padronizada: True
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-

## Padronização das datas

Antes da consolidação definitiva é necessário tratar o formato textual da data, que varia entre os arquivos. Para evidenciar o problema, faz-se primeiro uma consolidação simples e uma conversão direta das datas.

In [19]:
bases = []

for arquivo in arquivos:
    # Lê o arquivo e remove a coluna vazia
    df = ler_csv_bronze(arquivo, remover_colunas_vazias=True)

    # Padroniza o esquema
    df = df.rename(columns=mapa_colunas)

    # Guarda a origem do arquivo
    df["arquivo_origem"] = arquivo.name

    bases.append(df)

df_consolidado = pd.concat(
    bases,
    ignore_index=True
)

print(f"Linhas consolidadas: {df_consolidado.shape[0]:,}")
print(f"Colunas: {df_consolidado.shape[1]}")

Linhas consolidadas: 140,256
Colunas: 20


In [20]:
df_consolidado["DATA (YYYY-MM-DD)"] = pd.to_datetime(
    df_consolidado["DATA (YYYY-MM-DD)"],
    errors="coerce"
)

print("Data mínima:", df_consolidado["DATA (YYYY-MM-DD)"].min())
print("Data máxima:", df_consolidado["DATA (YYYY-MM-DD)"].max())
print("Datas inválidas:", df_consolidado["DATA (YYYY-MM-DD)"].isna().sum())

Data mínima: 2010-01-01 00:00:00
Data máxima: 2018-12-31 00:00:00
Datas inválidas: 61368


In [21]:
registros_por_ano = (
    df_consolidado
    .assign(ano=df_consolidado["DATA (YYYY-MM-DD)"].dt.year)
    .groupby("ano")
    .size()
)

print(registros_por_ano)

ano
2010.0    8760
2011.0    8760
2012.0    8784
2013.0    8760
2014.0    8760
2015.0    8760
2016.0    8784
2017.0    8760
2018.0    8760
dtype: int64


A conversão direta interpretou corretamente apenas os anos de 2010 a 2018, gerando datas inválidas e perda de registros nos demais anos. Inspecionando o primeiro registro de data de cada arquivo, observa-se que o formato muda ao longo do período (por exemplo, `YYYY-MM-DD`, `YYYY/MM/DD` e `DD/MM/YYYY`). O formato é então identificado por arquivo e aplicado na conversão.

In [22]:
for arquivo in arquivos:
    df_temp = ler_csv_bronze(arquivo, remover_colunas_vazias=True)
    df_temp = df_temp.rename(columns=mapa_colunas)

    print(
        arquivo.name,
        "->",
        df_temp["DATA (YYYY-MM-DD)"].iloc[0]
    )

INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2010_A_31-12-2010.CSV -> 2010-01-01
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2011_A_31-12-2011.CSV -> 2011-01-01


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2012_A_31-12-2012.CSV -> 2012-01-01


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2013_A_31-12-2013.CSV -> 2013-01-01


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2014_A_31-12-2014.CSV -> 2014-01-01
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2015_A_31-12-2015.CSV -> 2015-01-01


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2016_A_31-12-2016.CSV -> 2016-01-01
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2017_A_31-12-2017.CSV -> 2017-01-01


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2018_A_31-12-2018.CSV -> 2018-01-01


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2019_A_31-12-2019.CSV -> 2019/01/01
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2020_A_31-12-2020.CSV -> 2020/01/01
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2021_A_31-12-2021.csv -> 01/01/2021


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2022_A_31-12-2022.CSV -> 2022/01/01
INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2023_A_31-12-2023.CSV -> 2023/01/01


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2024_A_31-12-2024.CSV -> 2024/01/01


INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2025_A_31-12-2025.CSV -> 2025/01/01


## Consolidação

Com o formato de data identificado por arquivo, consolida-se a base horária de 2010 a 2025 aplicando o esquema padronizado e a conversão de datas específica de cada arquivo.

Os valores originais são preservados, incluindo códigos especiais como `-9999`. O tratamento de qualidade será realizado posteriormente, em etapa específica.

In [23]:
bases = []

for arquivo in arquivos:
    # Lê o arquivo e remove a coluna vazia
    df = ler_csv_bronze(arquivo, remover_colunas_vazias=True)

    # Padroniza nomes das colunas
    df = df.rename(columns=mapa_colunas)

    # Padroniza a data de acordo com o formato encontrado
    primeira_data = str(df["DATA (YYYY-MM-DD)"].iloc[0])

    if "-" in primeira_data:
        formato_data = "%Y-%m-%d"

    elif primeira_data.startswith("01/01/2021"):
        formato_data = "%d/%m/%Y"

    else:
        formato_data = "%Y/%m/%d"

    df["DATA (YYYY-MM-DD)"] = pd.to_datetime(
        df["DATA (YYYY-MM-DD)"],
        format=formato_data,
        errors="coerce"
    )

    # Rastreabilidade
    df["arquivo_origem"] = arquivo.name

    bases.append(df)

df_consolidado = pd.concat(
    bases,
    ignore_index=True
)

## Validação da base consolidada

Verificação do intervalo de datas (mínima, máxima e datas inválidas) e da contagem de registros por ano, confirmando que a base cobre integralmente o período de 2010 a 2025.

In [24]:
print("Data mínima:", df_consolidado["DATA (YYYY-MM-DD)"].min())
print("Data máxima:", df_consolidado["DATA (YYYY-MM-DD)"].max())
print("Datas inválidas:", df_consolidado["DATA (YYYY-MM-DD)"].isna().sum())

Data mínima: 2010-01-01 00:00:00
Data máxima: 2025-12-31 00:00:00
Datas inválidas: 0


In [25]:
registros_por_ano = (
    df_consolidado
    .assign(ano=df_consolidado["DATA (YYYY-MM-DD)"].dt.year)
    .groupby("ano")
    .size()
)

print(registros_por_ano)

ano
2010    8760
2011    8760
2012    8784
2013    8760
2014    8760
2015    8760
2016    8784
2017    8760
2018    8760
2019    8760
2020    8784
2021    8760
2022    8760
2023    8760
2024    8784
2025    8760
dtype: int64


## Exportação para Silver

Gravação da base horária consolidada na camada `silver`, mantendo os valores originais. Esse arquivo é a entrada das próximas etapas de tratamento.

In [26]:
from pathlib import Path

pasta_silver = Path("../data/silver")
pasta_silver.mkdir(parents=True, exist_ok=True)

caminho_saida = pasta_silver / "a701_2010_2025_horario_consolidado.csv"

df_consolidado.to_csv(
    caminho_saida,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

print(f"Arquivo salvo em: {caminho_saida}")

Arquivo salvo em: ..\data\silver\a701_2010_2025_horario_consolidado.csv


## Síntese

- Foram identificados e validados os 16 arquivos anuais da estação A701, cobrindo o período de 2010 a 2025 sem lacunas.
- A estrutura das 19 variáveis é consistente; apenas a nomenclatura de algumas colunas mudou ao longo dos anos, o que foi padronizado na leitura.
- O formato textual das datas também variava entre os arquivos e foi tratado por arquivo antes da consolidação.
- A base horária consolidada (140.256 registros) foi validada e exportada para a camada `silver`, preservando os valores originais, inclusive os códigos `-9999`.
- O tratamento dos códigos `-9999` e as demais etapas de qualidade serão realizados nos próximos notebooks.